# LeetCode 721 — Accounts Merge bằng Python

Link đề: https://leetcode.com/problems/accounts-merge/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Union Find, Array, Hash Table, String, Sorting

Skeleton LeetCode:

```python
class Solution:
    def accountsMerge(self, accounts: List[List[str]]) -> List[List[str]]:
        ...
```

Ghi chú: đề cho phép trả các tài khoản **theo thứ tự bất kỳ**, nên trong notebook mọi phép so sánh đều **chuẩn hóa** kết quả (sắp xếp danh sách các tài khoản) trước khi so với đáp án.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: gộp các tài khoản có **ít nhất một email chung** (kể cả gián tiếp: A chung với B, B chung với C thì A, B, C là một người); kết quả mỗi người gồm tên + các email **đã sắp xếp**.
2. Nắm các công cụ Python: giải nén `name, *emails = account`, `set` và phép hợp `|`, giao `&`, `defaultdict(set/list)`, `sorted`, ngăn xếp/hàng đợi cho DFS/BFS, và cấu trúc **Union-Find** (Disjoint Set Union).
3. Đi từ **gộp lặp đi lặp lại cho tới khi không gộp được nữa** → **Union-Find trên chỉ số tài khoản** → **DFS trên đồ thị email** → **BFS trên đồ thị tài khoản-email**.
4. Hiểu mấu chốt: đây là bài **thành phần liên thông (connected components)**. Hai email trong cùng một tài khoản được nối với nhau; mỗi thành phần liên thông là một người.
5. Tránh các lỗi kinh điển: gộp theo **tên** (hai người khác nhau có thể trùng tên), gộp một lượt không đệ quy/bắc cầu, dùng `parent[i]` thay vì `find(i)`, quên sắp xếp/loại trùng email, và so sánh kết quả mà không chuẩn hóa thứ tự.

## 2. Đề bài & ví dụ

Cho danh sách `accounts`, mỗi phần tử `accounts[i]` là một list chuỗi: phần tử đầu `accounts[i][0]` là **tên**, các phần tử còn lại là **email** của tài khoản đó.

Hai tài khoản **chắc chắn cùng một người** nếu có **ít nhất một email chung**. Chú ý: hai tài khoản trùng tên vẫn có thể là hai người khác nhau. Một người có thể có nhiều tài khoản, nhưng mọi tài khoản của họ đều cùng tên.

Sau khi gộp, trả về các tài khoản theo định dạng: phần tử đầu là tên, các phần tử sau là email **theo thứ tự đã sắp xếp**. Các tài khoản có thể trả về theo **thứ tự bất kỳ**.

**Ví dụ 1**

- Input: `accounts = [["John","johnsmith@mail.com","john_newyork@mail.com"],["John","johnsmith@mail.com","john00@mail.com"],["Mary","mary@mail.com"],["John","johnnybravo@mail.com"]]`
- Output: `[["John","john00@mail.com","john_newyork@mail.com","johnsmith@mail.com"],["Mary","mary@mail.com"],["John","johnnybravo@mail.com"]]`
- Giải thích: hai tài khoản John đầu có chung `johnsmith@mail.com` nên là một người. John thứ ba và Mary không chung email với ai nên đứng riêng. Có thể trả về theo thứ tự khác.

**Ví dụ 2**

- Input: `accounts = [["Gabe","Gabe0@m.co","Gabe3@m.co","Gabe1@m.co"],["Kevin","Kevin3@m.co","Kevin5@m.co","Kevin0@m.co"],["Ethan","Ethan5@m.co","Ethan4@m.co","Ethan0@m.co"],["Hanzo","Hanzo3@m.co","Hanzo1@m.co","Hanzo0@m.co"],["Fern","Fern5@m.co","Fern1@m.co","Fern0@m.co"]]`
- Output: mỗi tài khoản giữ nguyên nhưng email được sắp xếp, ví dụ `["Gabe","Gabe0@m.co","Gabe1@m.co","Gabe3@m.co"]`, ...

**Ràng buộc:**

- `1 <= accounts.length <= 1000`
- `2 <= accounts[i].length <= 10`
- `1 <= accounts[i][j].length <= 30`
- `accounts[i][0]` chỉ gồm chữ cái tiếng Anh
- `accounts[i][j]` (với `j > 0`) là email hợp lệ

Bài này có **nhiều đáp án đúng về thứ tự**, nên test bằng cách **chuẩn hóa** (sắp xếp các tài khoản kết quả) rồi so sánh.

In [1]:
from typing import List
from collections import defaultdict, deque
import random

def norm(result):
    """Chuẩn hóa kết quả: mỗi tài khoản thành tuple, rồi sắp xếp danh sách (thứ tự trả về là tùy ý)."""
    return sorted(tuple(a) for a in result)

ex1_in = [["John", "johnsmith@mail.com", "john_newyork@mail.com"],
          ["John", "johnsmith@mail.com", "john00@mail.com"],
          ["Mary", "mary@mail.com"],
          ["John", "johnnybravo@mail.com"]]
ex1_out = [["John", "john00@mail.com", "john_newyork@mail.com", "johnsmith@mail.com"],
           ["Mary", "mary@mail.com"],
           ["John", "johnnybravo@mail.com"]]
ex2_in = [["Gabe", "Gabe0@m.co", "Gabe3@m.co", "Gabe1@m.co"],
          ["Kevin", "Kevin3@m.co", "Kevin5@m.co", "Kevin0@m.co"],
          ["Ethan", "Ethan5@m.co", "Ethan4@m.co", "Ethan0@m.co"],
          ["Hanzo", "Hanzo3@m.co", "Hanzo1@m.co", "Hanzo0@m.co"],
          ["Fern", "Fern5@m.co", "Fern1@m.co", "Fern0@m.co"]]
ex2_out = [["Ethan", "Ethan0@m.co", "Ethan4@m.co", "Ethan5@m.co"],
           ["Gabe", "Gabe0@m.co", "Gabe1@m.co", "Gabe3@m.co"],
           ["Hanzo", "Hanzo0@m.co", "Hanzo1@m.co", "Hanzo3@m.co"],
           ["Kevin", "Kevin0@m.co", "Kevin3@m.co", "Kevin5@m.co"],
           ["Fern", "Fern0@m.co", "Fern1@m.co", "Fern5@m.co"]]
examples = [(ex1_in, ex1_out), (ex2_in, ex2_out)]

for accounts, expected in examples:
    print(len(accounts), "tài khoản ->", len(expected), "người")
print(norm(ex1_out)[0])

4 tài khoản -> 3 người
5 tài khoản -> 5 người
('John', 'john00@mail.com', 'john_newyork@mail.com', 'johnsmith@mail.com')


## 3. Kiến thức nền

### 3.1. Giải nén `name, *emails = account`

Dấu `*` trong phép gán gom **phần còn lại** vào một list: `name, *emails = ["John", "a@x", "b@x"]` cho `name = "John"`, `emails = ["a@x", "b@x"]`. Ta cũng dùng được ngay trong vòng `for`: `for name, *emails in accounts`.

In [2]:
account = ["John", "johnsmith@mail.com", "john00@mail.com"]
name, *emails = account
print(name, emails)
for name, *emails in ex1_in:
    print(f"{name}: {len(emails)} email")

John ['johnsmith@mail.com', 'john00@mail.com']
John: 2 email
John: 2 email
Mary: 1 email
John: 1 email


### 3.2. `set` và các phép hợp/giao

`set` không chứa phần tử trùng. `a | b` là hợp, `a & b` là giao (các phần tử chung). Kiểm tra "hai tài khoản có email chung không" = giao khác rỗng: `bool(a & b)`. `sorted(s)` biến set thành list đã sắp xếp (theo thứ tự chuỗi).

In [3]:
a = {"x@m", "y@m"}
b = {"y@m", "z@m"}
c = {"p@m"}
print(a & b, a | b, bool(a & c))
print(sorted(a | b | c))
print(sorted({"b@m", "a@m", "B@m"}))      # chữ HOA đứng trước chữ thường theo mã ASCII

{'y@m'} {'z@m', 'y@m', 'x@m'} False
['p@m', 'x@m', 'y@m', 'z@m']
['B@m', 'a@m', 'b@m']


### 3.3. `defaultdict` — dict có giá trị mặc định

`defaultdict(list)` tự tạo `[]` khi truy cập khóa chưa có; `defaultdict(set)` tạo `set()`. Rất tiện cho việc nhóm: `g[key].append(x)` không cần kiểm tra khóa tồn tại trước.

In [4]:
g = defaultdict(list)
for i, (name, *emails) in enumerate(ex1_in):
    for e in emails:
        g[e].append(i)
print(dict(g))
print([e for e, idx in g.items() if len(idx) > 1], "<- email xuất hiện ở nhiều tài khoản")

{'johnsmith@mail.com': [0, 1], 'john_newyork@mail.com': [0], 'john00@mail.com': [1], 'mary@mail.com': [2], 'johnnybravo@mail.com': [3]}
['johnsmith@mail.com'] <- email xuất hiện ở nhiều tài khoản


### 3.4. Thành phần liên thông (connected components)

Hình dung đồ thị: mỗi **tài khoản** là một đỉnh; vẽ cạnh giữa hai tài khoản nếu chúng có email chung. Một **thành phần liên thông** là nhóm đỉnh mà từ đỉnh này đi tới đỉnh kia qua các cạnh. "Cùng một người" chính là "cùng một thành phần liên thông" — kể cả khi A và C **không** chung email trực tiếp mà cùng chung với B (quan hệ **bắc cầu**).

```
A --b-- B --c-- C        => A, B, C là một người (dù A và C không chung email nào)
```

In [5]:
accs = [["Al", "a", "b"], ["Al", "b", "c"], ["Al", "c", "d"], ["Bo", "z"]]
sets = [set(x[1:]) for x in accs]
for i in range(len(accs)):
    for j in range(i + 1, len(accs)):
        print(f"tài khoản {i} & {j}: chung {sorted(sets[i] & sets[j])}")
print("0 và 2 không chung trực tiếp, nhưng cùng liên thông qua 1")

tài khoản 0 & 1: chung ['b']
tài khoản 0 & 2: chung []
tài khoản 0 & 3: chung []
tài khoản 1 & 2: chung ['c']
tài khoản 1 & 3: chung []
tài khoản 2 & 3: chung []
0 và 2 không chung trực tiếp, nhưng cùng liên thông qua 1


### 3.5. Union-Find (Disjoint Set Union)

**Union-Find** quản lý các **tập rời nhau**. Mỗi phần tử có một "cha" (`parent`); phần tử là **gốc (root)** khi cha là chính nó. Hai thao tác:

- `find(x)`: đi theo cha cho tới gốc, trả về gốc đại diện của tập chứa `x`. **Nén đường đi (path compression)**: gán luôn cha của mọi nút đã đi qua thành gốc để lần sau nhanh.
- `union(a, b)`: nối hai tập: đặt gốc của một tập làm con của gốc tập kia (nối tập nhỏ vào tập lớn: **union by size**).

Hai phần tử cùng tập ⇔ `find(a) == find(b)`. Thời gian mỗi thao tác gần như `O(1)` (chính xác `O(α(n))`, hàm Ackermann ngược).

In [6]:
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))        # ban đầu mỗi phần tử là một tập riêng, cha là chính nó
        self.size = [1] * n

    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])      # nén đường đi
        return self.p[x]

    def union(self, a, b):
        pa, pb = self.find(a), self.find(b)
        if pa == pb:
            return False
        if self.size[pa] > self.size[pb]:
            self.p[pb] = pa
            self.size[pa] += self.size[pb]
        else:
            self.p[pa] = pb
            self.size[pb] += self.size[pa]
        return True

uf = UnionFind(6)
print("ban đầu p =", uf.p)
uf.union(0, 1); uf.union(2, 3); uf.union(1, 3)
print("sau union(0,1), union(2,3), union(1,3): p =", uf.p)
print("find(0) == find(2)?", uf.find(0) == uf.find(2), "| find(0) == find(5)?", uf.find(0) == uf.find(5))
print("gốc của từng phần tử:", [uf.find(i) for i in range(6)])

ban đầu p = [0, 1, 2, 3, 4, 5]
sau union(0,1), union(2,3), union(1,3): p = [1, 3, 3, 3, 4, 5]
find(0) == find(2)? True | find(0) == find(5)? False
gốc của từng phần tử: [3, 3, 3, 3, 4, 5]


### 3.6. DFS với ngăn xếp và BFS với hàng đợi

DFS lặp: `stack = [start]`, `pop()` lấy phần tử cuối, đẩy láng giềng chưa thăm. BFS: `deque([start])`, `popleft()` lấy đầu. Cả hai thăm **mọi đỉnh trong cùng thành phần liên thông** của `start`; thứ tự khác nhau nhưng tập đỉnh thăm được **giống nhau**. Luôn đánh dấu "đã thăm" khi **đưa vào** cấu trúc để không thăm lặp.

In [7]:
graph = {1: [2], 2: [1, 3], 3: [2], 4: [5], 5: [4]}
def component(start, use_stack=True):
    seen = {start}
    todo = [start] if use_stack else deque([start])
    order = []
    while todo:
        u = todo.pop() if use_stack else todo.popleft()
        order.append(u)
        for v in graph[u]:
            if v not in seen:
                seen.add(v)
                todo.append(v)
    return order

print("DFS từ 1:", component(1, True), "| BFS từ 1:", component(1, False))
print("thành phần của 4:", component(4))

DFS từ 1: [1, 2, 3] | BFS từ 1: [1, 2, 3]
thành phần của 4: [4, 5]


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 (đánh số tài khoản `0..3`):

- Tài khoản 0: `johnsmith`, `john_newyork`. Tài khoản 1: `johnsmith`, `john00`. **Chung `johnsmith`** → 0 và 1 là một người.
- Tài khoản 2 (`mary`) và 3 (`johnnybravo`) không chung email với ai.
- Người thứ nhất: tên John, email = `{johnsmith, john_newyork, john00}` → sắp xếp: `john00, john_newyork, johnsmith`.

Hãy để ý hai điều dễ nhầm: (a) tài khoản 3 cũng tên John nhưng **không** gộp (khác email); (b) kết quả phải **sắp xếp** email.

Bài toán thực chất: **gom nhóm theo quan hệ bắc cầu "có chung email"**, rồi mỗi nhóm hợp các email và sắp xếp. Cách "ngây thơ" nhất: lặp lại việc gộp hai nhóm bất kỳ có giao khác rỗng, cho đến khi không gộp được nữa.

In [8]:
def show_groups(accounts):
    sets = [set(a[1:]) for a in accounts]
    for i in range(len(accounts)):
        for j in range(i + 1, len(accounts)):
            inter = sets[i] & sets[j]
            if inter:
                print(f"tài khoản {i} ({accounts[i][0]}) và {j} ({accounts[j][0]}) chung {sorted(inter)}")

show_groups(ex1_in)

tài khoản 0 (John) và 1 (John) chung ['johnsmith@mail.com']


## 5. Cách 1 — Brute Force: gộp lặp lại cho đến khi không gộp được

### Ý tưởng

Giữ danh sách các nhóm `(tên, tập email)`. Lặp: tìm **một cặp** nhóm có giao khác rỗng, gộp chúng (hợp tập email), và bắt đầu lại; dừng khi quét hết mà không gộp được cặp nào. Cuối cùng, mỗi nhóm → `[tên] + sorted(email)`.

### Code

In [9]:
def merge_brute(accounts):
    groups = [(a[0], set(a[1:])) for a in accounts]
    merged = True
    while merged:
        merged = False
        for i in range(len(groups)):
            for j in range(i + 1, len(groups)):
                if groups[i][1] & groups[j][1]:
                    groups[i] = (groups[i][0], groups[i][1] | groups[j][1])
                    groups.pop(j)
                    merged = True
                    break
            if merged:
                break
    return [[name] + sorted(emails) for name, emails in groups]

for accounts, expected in examples:
    got = merge_brute(accounts)
    print(len(got), "người")
    assert norm(got) == norm(expected)

3 người
5 người


### Độ phức tạp và nút thắt

- Mỗi lần gộp giảm số nhóm đi 1, tối đa `N - 1` lần gộp (`N` là số tài khoản). Mỗi vòng quét tìm cặp tốn `O(N^2)` phép giao tập, mỗi giao `O(E)` (`E <= 10`).
- Time tệ nhất `O(N^3 * E)` ≈ `10^10` với `N = 1000` — quá chậm. Space `O(N * E)`.

**Nút thắt:** sau mỗi lần gộp ta **bắt đầu quét lại từ đầu** và lại so mọi cặp. Ta muốn tìm "tài khoản nào chung email với tài khoản nào" **trực tiếp** bằng tra cứu email, không so từng cặp.

In [10]:
def brute_pair_checks(accounts):
    """Đếm số lần kiểm tra giao của hai nhóm khi chạy merge_brute."""
    groups = [set(a[1:]) for a in accounts]
    checks = 0
    merged = True
    while merged:
        merged = False
        for i in range(len(groups)):
            for j in range(i + 1, len(groups)):
                checks += 1
                if groups[i] & groups[j]:
                    groups[i] |= groups[j]
                    groups.pop(j)
                    merged = True
                    break
            if merged:
                break
    return checks

# chuỗi: tài khoản i chung email với tài khoản i+1 (xấu cho brute)
for n in [10, 20, 40, 80]:
    chain = [["P", f"e{i}", f"e{i + 1}"] for i in range(n)][::-1]
    print(f"n={n}: số lần kiểm tra giao = {brute_pair_checks(chain)}")

n=10: số lần kiểm tra giao = 9
n=20: số lần kiểm tra giao = 19
n=40: số lần kiểm tra giao = 39
n=80: số lần kiểm tra giao = 79


## 6. Ý tưởng thuật toán / bước đột phá

**Dùng email làm "khóa nối".** Duyệt tài khoản một lần, và dùng một dict `email → chỉ số tài khoản đầu tiên chứa email này`:

- Gặp email chưa thấy: ghi `d[email] = i`.
- Gặp email **đã thấy** ở tài khoản `j = d[email]`: tài khoản `i` và `j` chung email → **nối** `i` với `j` (union).

Như vậy mọi cặp tài khoản chung email đều được nối **gián tiếp** qua tài khoản đầu tiên chứa email đó, mà **không phải so từng cặp**. Quan hệ bắc cầu do **Union-Find** (hoặc duyệt đồ thị) lo.

Cuối cùng gom: với mỗi tài khoản `i`, tìm gốc `find(i)` rồi thêm email của `i` vào tập của gốc đó. Mỗi gốc → một người: `[tên của gốc] + sorted(emails)`.

In [11]:
# Quan sát bước nối: d[email] = tài khoản đầu tiên; mỗi lần gặp lại thì nối
uf_demo = UnionFind(len(ex1_in))
d = {}
for i, (name, *emails) in enumerate(ex1_in):
    for e in emails:
        if e in d:
            print(f"tài khoản {i} gặp lại {e!r} (đã ở tài khoản {d[e]}) -> union({i}, {d[e]})")
            uf_demo.union(i, d[e])
        else:
            d[e] = i
print("gốc của từng tài khoản:", [uf_demo.find(i) for i in range(len(ex1_in))])

tài khoản 1 gặp lại 'johnsmith@mail.com' (đã ở tài khoản 0) -> union(1, 0)
gốc của từng tài khoản: [0, 0, 2, 3]


## 7. Các cách tiếp theo

Phần này có thêm 3 cách (tổng 4 cách): **Union-Find trên chỉ số tài khoản** (lời giải nộp), **DFS trên đồ thị email**, **BFS trên đồ thị tài khoản-email**.

### 7.1. Cách 2 — Union-Find trên chỉ số tài khoản (lời giải nộp)

**Giải thích từng đoạn.**

1. Tạo `UnionFind(n)` với `n = len(accounts)` (mỗi tài khoản một tập).
2. Dict `d`: email → chỉ số tài khoản đầu tiên chứa nó. Duyệt tài khoản `i` và email `e`: nếu `e in d` thì `uf.union(i, d[e])`, ngược lại `d[e] = i`.
3. `g = defaultdict(set)`: với mỗi `i`, `g[uf.find(i)].update(emails của i)`.
4. Kết quả: với mỗi `(root, emails)`: `[accounts[root][0]] + sorted(emails)`. Tên của `root` đúng vì mọi tài khoản trong một nhóm cùng tên.

In [12]:
def merge_uf(accounts):
    uf = UnionFind(len(accounts))
    d = {}
    for i, (_, *emails) in enumerate(accounts):
        for e in emails:
            if e in d:
                uf.union(i, d[e])
            else:
                d[e] = i
    g = defaultdict(set)
    for i, (_, *emails) in enumerate(accounts):
        g[uf.find(i)].update(emails)
    return [[accounts[root][0]] + sorted(emails) for root, emails in g.items()]

for accounts, expected in examples:
    got = merge_uf(accounts)
    print(len(got), "người")
    assert norm(got) == norm(expected)

3 người
5 người


In [13]:
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))
        self.size = [1] * n

    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]

    def union(self, a, b):
        pa, pb = self.find(a), self.find(b)
        if pa == pb:
            return False
        if self.size[pa] > self.size[pb]:
            self.p[pb] = pa
            self.size[pa] += self.size[pb]
        else:
            self.p[pa] = pb
            self.size[pb] += self.size[pa]
        return True


class Solution:
    def accountsMerge(self, accounts: List[List[str]]) -> List[List[str]]:
        uf = UnionFind(len(accounts))
        d = {}
        for i, (_, *emails) in enumerate(accounts):
            for email in emails:
                if email in d:
                    uf.union(i, d[email])
                else:
                    d[email] = i
        g = defaultdict(set)
        for i, (_, *emails) in enumerate(accounts):
            g[uf.find(i)].update(emails)
        return [[accounts[root][0]] + sorted(emails) for root, emails in g.items()]

print(norm(Solution().accountsMerge(ex1_in)) == norm(ex1_out))

True


**Độ phức tạp:** gọi `K` là tổng số email (`<= 10^4`). Duyệt + union `O(K α(N))`, gom `O(K)`, sắp xếp email mỗi nhóm `O(K log K)` tổng cộng → Time `O(K log K)`, Space `O(K + N)`.

### 7.2. Cách 3 — DFS trên đồ thị email

**Ý tưởng.** Dựng đồ thị **đỉnh là email**: trong mỗi tài khoản, nối email đầu tiên với mọi email còn lại (đủ để mọi email của tài khoản liên thông). Ghi `owner[email] = tên`. Mỗi **thành phần liên thông** của đồ thị này là một người; DFS thu thập các email, rồi sắp xếp.

**Giải thích từng đoạn.**

1. `graph = defaultdict(set)`: với mỗi tài khoản, `graph[first].add(e)` và `graph[e].add(first)`.
2. `owner[e] = name`.
3. Với mỗi email chưa thăm, DFS bằng ngăn xếp, gom `comp`, kết quả `[owner[start]] + sorted(comp)`.

In [14]:
def merge_dfs(accounts):
    graph = defaultdict(set)
    owner = {}
    for name, *emails in accounts:
        first = emails[0]
        for e in emails:
            owner[e] = name
            graph[first].add(e)
            graph[e].add(first)
    seen = set()
    result = []
    for start in owner:
        if start in seen:
            continue
        seen.add(start)
        stack = [start]
        comp = []
        while stack:
            u = stack.pop()
            comp.append(u)
            for v in graph[u]:
                if v not in seen:
                    seen.add(v)
                    stack.append(v)
        result.append([owner[start]] + sorted(comp))
    return result

for accounts, expected in examples:
    got = merge_dfs(accounts)
    print(len(got), "người")
    assert norm(got) == norm(expected)

3 người
5 người


In [15]:
class Solution:
    def accountsMerge(self, accounts: List[List[str]]) -> List[List[str]]:
        graph = defaultdict(set)
        owner = {}
        for name, *emails in accounts:
            first = emails[0]
            for e in emails:
                owner[e] = name
                graph[first].add(e)
                graph[e].add(first)
        seen = set()
        result = []
        for start in owner:
            if start in seen:
                continue
            seen.add(start)
            stack = [start]
            comp = []
            while stack:
                u = stack.pop()
                comp.append(u)
                for v in graph[u]:
                    if v not in seen:
                        seen.add(v)
                        stack.append(v)
            result.append([owner[start]] + sorted(comp))
        return result

print(norm(Solution().accountsMerge(ex2_in)) == norm(ex2_out))

True


**Độ phức tạp:** đồ thị có `K` đỉnh và `O(K)` cạnh; DFS `O(K)`, sắp xếp `O(K log K)` → Time `O(K log K)`, Space `O(K)`. Hai email trong cùng tài khoản mà trùng tên: vì tên chung nên `owner` luôn đúng.

### 7.3. Cách 4 — BFS trên đồ thị tài khoản-email (hai phía)

**Ý tưởng.** Dựng đồ thị **hai phía**: một bên là tài khoản, bên kia là email; cạnh nối tài khoản với từng email của nó. Từ một tài khoản chưa thăm, BFS: lấy các email của nó, từ mỗi email lấy mọi tài khoản chứa email đó (qua dict `email → danh sách tài khoản`), thêm các tài khoản chưa thăm vào hàng đợi. Các email thu được là của một người.

In [16]:
def merge_bfs(accounts):
    email_to_accounts = defaultdict(list)
    for i, (_, *emails) in enumerate(accounts):
        for e in emails:
            email_to_accounts[e].append(i)
    visited = [False] * len(accounts)
    result = []
    for s in range(len(accounts)):
        if visited[s]:
            continue
        visited[s] = True
        q = deque([s])
        emails_of_person = set()
        while q:
            i = q.popleft()
            for e in accounts[i][1:]:
                emails_of_person.add(e)
                for j in email_to_accounts[e]:
                    if not visited[j]:
                        visited[j] = True
                        q.append(j)
        result.append([accounts[s][0]] + sorted(emails_of_person))
    return result

for accounts, expected in examples:
    got = merge_bfs(accounts)
    print(len(got), "người")
    assert norm(got) == norm(expected)

3 người
5 người


In [17]:
class Solution:
    def accountsMerge(self, accounts: List[List[str]]) -> List[List[str]]:
        email_to_accounts = defaultdict(list)
        for i, (_, *emails) in enumerate(accounts):
            for e in emails:
                email_to_accounts[e].append(i)
        visited = [False] * len(accounts)
        result = []
        for s in range(len(accounts)):
            if visited[s]:
                continue
            visited[s] = True
            q = deque([s])
            person = set()
            while q:
                i = q.popleft()
                for e in accounts[i][1:]:
                    person.add(e)
                    for j in email_to_accounts[e]:
                        if not visited[j]:
                            visited[j] = True
                            q.append(j)
            result.append([accounts[s][0]] + sorted(person))
        return result

print(norm(Solution().accountsMerge(ex1_in)) == norm(ex1_out))

True


**Độ phức tạp:** mỗi tài khoản vào hàng đợi một lần; mỗi (tài khoản, email) được xét `O(1)` lần cộng với duyệt danh sách tài khoản của email → tổng `O(K)`; sắp xếp `O(K log K)`. Time `O(K log K)`, Space `O(K + N)`.

## 8. TRACE từng bước

### 8.1. Trace Union-Find với ví dụ 1

`accounts` (chỉ số): `0: John [johnsmith, john_newyork]`, `1: John [johnsmith, john00]`, `2: Mary [mary]`, `3: John [johnnybravo]`.

| tài khoản `i` | email `e` | `e` đã trong `d`? | hành động | `p` sau đó |
|---|---|---|---|---|
| 0 | johnsmith | không | `d[johnsmith] = 0` | `[0,1,2,3]` |
| 0 | john_newyork | không | `d[john_newyork] = 0` | `[0,1,2,3]` |
| 1 | johnsmith | **có** (0) | `union(1, 0)` | `[0,0,2,3]` |
| 1 | john00 | không | `d[john00] = 1` | như trên |
| 2 | mary | không | `d[mary] = 2` | như trên |
| 3 | johnnybravo | không | `d[johnnybravo] = 3` | như trên |

Ở `union(1, 0)`: hai tập cùng kích thước 1 nên nhánh `else` chạy, `p[1] = 0` (gốc là `0`). Sau đó gom theo gốc: tài khoản 0 và 1 cùng gốc → một người; 2 và 3 mỗi cái một người.

In [18]:
def trace_uf(accounts):
    uf = UnionFind(len(accounts))
    d = {}
    for i, (name, *emails) in enumerate(accounts):
        for e in emails:
            if e in d:
                uf.union(i, d[e])
                print(f"tài khoản {i}, {e!r}: đã thấy ở {d[e]} -> union({i}, {d[e]}) | p = {uf.p} | size = {uf.size}")
            else:
                d[e] = i
                print(f"tài khoản {i}, {e!r}: mới -> d[{e!r}] = {i}")
    g = defaultdict(set)
    for i, (_, *emails) in enumerate(accounts):
        g[uf.find(i)].update(emails)
    print("gom theo gốc:", {root: sorted(es) for root, es in g.items()})
    return [[accounts[root][0]] + sorted(es) for root, es in g.items()]

res = trace_uf(ex1_in)
assert norm(res) == norm(ex1_out)

tài khoản 0, 'johnsmith@mail.com': mới -> d['johnsmith@mail.com'] = 0
tài khoản 0, 'john_newyork@mail.com': mới -> d['john_newyork@mail.com'] = 0
tài khoản 1, 'johnsmith@mail.com': đã thấy ở 0 -> union(1, 0) | p = [0, 0, 2, 3] | size = [2, 1, 1, 1]
tài khoản 1, 'john00@mail.com': mới -> d['john00@mail.com'] = 1
tài khoản 2, 'mary@mail.com': mới -> d['mary@mail.com'] = 2
tài khoản 3, 'johnnybravo@mail.com': mới -> d['johnnybravo@mail.com'] = 3
gom theo gốc: {0: ['john00@mail.com', 'john_newyork@mail.com', 'johnsmith@mail.com'], 2: ['mary@mail.com'], 3: ['johnnybravo@mail.com']}


### 8.2. Trace Union-Find với quan hệ bắc cầu

`accounts = [["A","e1"], ["A","e2"], ["A","e1","e2"], ["B","e9"]]`. Tài khoản 2 nối 0 (qua `e1`) rồi nối 1 (qua `e2`), nên 0, 1, 2 thành một người dù 0 và 1 không chung email trực tiếp.

In [19]:
res = trace_uf([["A", "e1"], ["A", "e2"], ["A", "e1", "e2"], ["B", "e9"]])
print(norm(res))
assert norm(res) == [("A", "e1", "e2"), ("B", "e9")]

tài khoản 0, 'e1': mới -> d['e1'] = 0
tài khoản 1, 'e2': mới -> d['e2'] = 1
tài khoản 2, 'e1': đã thấy ở 0 -> union(2, 0) | p = [0, 1, 0, 3] | size = [2, 1, 1, 1]
tài khoản 2, 'e2': đã thấy ở 1 -> union(2, 1) | p = [0, 0, 0, 3] | size = [3, 1, 1, 1]
tài khoản 3, 'e9': mới -> d['e9'] = 3
gom theo gốc: {0: ['e1', 'e2'], 3: ['e9']}
[('A', 'e1', 'e2'), ('B', 'e9')]


### 8.3. Trace DFS trên đồ thị email

Với ví dụ 1, đồ thị email: `johnsmith — john_newyork`, `johnsmith — john00` (tài khoản 0 và 1 cùng chứa `johnsmith`, nối chung nút này), `mary` (đơn độc), `johnnybravo` (đơn độc). DFS từ `johnsmith` (nút đầu của tài khoản 0 làm trung tâm) thu được 3 email.

In [20]:
def trace_dfs(accounts):
    graph = defaultdict(set)
    owner = {}
    for name, *emails in accounts:
        first = emails[0]
        for e in emails:
            owner[e] = name
            graph[first].add(e)
            graph[e].add(first)
    print("đồ thị email:", {k: sorted(v) for k, v in graph.items()})
    seen, result = set(), []
    for start in owner:
        if start in seen:
            continue
        seen.add(start)
        stack, comp = [start], []
        print(f"DFS từ {start!r}")
        while stack:
            u = stack.pop()
            comp.append(u)
            new = [v for v in sorted(graph[u]) if v not in seen]
            for v in new:
                seen.add(v)
                stack.append(v)
            print(f"   lấy {u!r}, đẩy thêm {new}, stack = {stack}")
        result.append([owner[start]] + sorted(comp))
    return result

res = trace_dfs(ex1_in)
assert norm(res) == norm(ex1_out)

đồ thị email: {'johnsmith@mail.com': ['john00@mail.com', 'john_newyork@mail.com', 'johnsmith@mail.com'], 'john_newyork@mail.com': ['johnsmith@mail.com'], 'john00@mail.com': ['johnsmith@mail.com'], 'mary@mail.com': ['mary@mail.com'], 'johnnybravo@mail.com': ['johnnybravo@mail.com']}
DFS từ 'johnsmith@mail.com'
   lấy 'johnsmith@mail.com', đẩy thêm ['john00@mail.com', 'john_newyork@mail.com'], stack = ['john00@mail.com', 'john_newyork@mail.com']
   lấy 'john_newyork@mail.com', đẩy thêm [], stack = ['john00@mail.com']
   lấy 'john00@mail.com', đẩy thêm [], stack = []
DFS từ 'mary@mail.com'
   lấy 'mary@mail.com', đẩy thêm [], stack = []
DFS từ 'johnnybravo@mail.com'
   lấy 'johnnybravo@mail.com', đẩy thêm [], stack = []


## 9. Code nộp LeetCode

Lời giải nộp là **Union-Find** (Cách 2). Trên LeetCode, đặt lớp `UnionFind` phía trên `Solution` (và `defaultdict` đã được import sẵn trong môi trường LeetCode; khi chạy ở máy cục bộ nhớ `from collections import defaultdict`).

In [21]:
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))
        self.size = [1] * n

    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]

    def union(self, a, b):
        pa, pb = self.find(a), self.find(b)
        if pa == pb:
            return False
        if self.size[pa] > self.size[pb]:
            self.p[pb] = pa
            self.size[pa] += self.size[pb]
        else:
            self.p[pa] = pb
            self.size[pb] += self.size[pa]
        return True


class Solution:
    def accountsMerge(self, accounts: List[List[str]]) -> List[List[str]]:
        uf = UnionFind(len(accounts))
        d = {}
        for i, (_, *emails) in enumerate(accounts):
            for email in emails:
                if email in d:
                    uf.union(i, d[email])
                else:
                    d[email] = i
        g = defaultdict(set)
        for i, (_, *emails) in enumerate(accounts):
            g[uf.find(i)].update(emails)
        return [[accounts[root][0]] + sorted(emails) for root, emails in g.items()]


sol = Solution()
for accounts, expected in examples:
    got = sol.accountsMerge(accounts)
    print(len(got), "người")
    assert norm(got) == norm(expected)       # đề cho phép thứ tự bất kỳ -> chuẩn hóa rồi so sánh

3 người
5 người


## 10. Vì sao đúng

**Mô hình.** Xét đồ thị `G` có đỉnh là các tài khoản `0..N-1`; nối `i — j` nếu chung ít nhất một email. Theo đề, hai tài khoản chung email là cùng một người, và quan hệ "cùng một người" là **bắc cầu**; vậy các người chính là các **thành phần liên thông** của `G`.

**Mệnh đề 1 (Union-Find tạo đúng thành phần liên thông).** Với mỗi email `e` xuất hiện ở các tài khoản `i1 < i2 < ... < ik` (theo thứ tự duyệt), thuật toán đặt `d[e] = i1` và gọi `union(i_t, i1)` cho `t = 2..k`. Khi đó mọi tài khoản chứa `e` được nối với `i1`, nên **cùng một tập**. Ngược lại, mọi lần `union` chỉ nối hai tài khoản đã chứa chung một email (cùng nối với `d[e]`), nên không gộp nhầm các tài khoản không liên thông. Do union-find duy trì đúng quan hệ liên thông bắc cầu, tập kết quả = thành phần liên thông của `G`.

**Mệnh đề 2 (kết quả mỗi nhóm).** Với mỗi gốc `r`, tập email là **hợp** các email của mọi tài khoản trong nhóm; `sorted` cho thứ tự yêu cầu; không trùng vì dùng `set`. Tên lấy từ `accounts[r][0]` vì mọi tài khoản của một người cùng tên (đảm bảo của đề). Mỗi email thuộc đúng một nhóm (nếu thuộc hai nhóm thì các tài khoản chứa nó đã được nối).

**Chuẩn hóa khi kiểm thử.** Vì thứ tự các nhóm và thứ tự `dict` không được đề quy định, ta so sánh `sorted(tuple(a) for a in kết_quả)`. Ta kiểm tra thêm các tính chất bất biến bằng code (xem mục 12): phân hoạch email, đã sắp xếp, tên khớp.

In [22]:
def is_valid_result(accounts, result):
    """Kiểm tra các tính chất BẮT BUỘC của một kết quả hợp lệ (độc lập với thứ tự):
       (1) mỗi nhóm: email sắp xếp, không trùng; (2) mọi email trong input xuất hiện trong đúng một nhóm;
       (3) mọi tài khoản input nằm trọn trong một nhóm, và tên khớp."""
    where = {}
    for gi, grp in enumerate(result):
        emails = grp[1:]
        if emails != sorted(set(emails)) or not emails:
            return False
        for e in emails:
            if e in where:
                return False                    # email thuộc hai nhóm
            where[e] = gi
    all_emails = {e for a in accounts for e in a[1:]}
    if set(where) != all_emails:
        return False
    for a in accounts:
        gids = {where[e] for e in a[1:]}
        if len(gids) != 1:
            return False                        # một tài khoản bị tách đôi
        if result[next(iter(gids))][0] != a[0]:
            return False                        # sai tên
    return True

for accounts, expected in examples:
    assert is_valid_result(accounts, expected)
    assert is_valid_result(accounts, Solution().accountsMerge(accounts))
# một kết quả sai: tách một người làm đôi
bad = [["John", "john00@mail.com"], ["John", "john_newyork@mail.com", "johnsmith@mail.com"],
       ["Mary", "mary@mail.com"], ["John", "johnnybravo@mail.com"]]
print("kết quả tách sai bị phát hiện:", not is_valid_result(ex1_in, bad))

kết quả tách sai bị phát hiện: True


Lưu ý: `is_valid_result` kiểm tra điều kiện **không bị tách quá mức** (mỗi tài khoản nằm trọn một nhóm), còn việc **không gộp thừa** được kiểm tra bằng so với oracle (Cách 1) ở mục 12.

## 11. Time / Space Complexity

Gọi `N` là số tài khoản (`<= 1000`), `K` tổng số email (`<= 10^4`).

| Cách | Time | Space |
|---|---|---|
| 1. Gộp lặp lại | `O(N^3 * E)` | `O(K)` |
| 2. Union-Find (nộp) | `O(K α(N) + K log K)` | `O(K + N)` |
| 3. DFS email | `O(K log K)` | `O(K)` |
| 4. BFS tài khoản-email | `O(K log K)` | `O(K + N)` |

Phần `K log K` đến từ việc **sắp xếp email** của từng người (bắt buộc theo đề). Ta đếm số lần `union` thực sự gộp được hai tập khác nhau để thấy chi phí.

In [23]:
def count_effective_unions(accounts):
    uf = UnionFind(len(accounts))
    d = {}
    attempts = effective = 0
    for i, (_, *emails) in enumerate(accounts):
        for e in emails:
            if e in d:
                attempts += 1
                effective += uf.union(i, d[e])
            else:
                d[e] = i
    people = len({uf.find(i) for i in range(len(accounts))})
    return attempts, effective, people

print("ví dụ 1:", count_effective_unions(ex1_in), "(thử nối, nối thật, số người)")
print("ví dụ 2:", count_effective_unions(ex2_in))
# số người = N - số lần nối thật
a, e, p = count_effective_unions(ex1_in)
assert p == len(ex1_in) - e

ví dụ 1: (1, 1, 3) (thử nối, nối thật, số người)
ví dụ 2: (0, 0, 5)


## 12. Bộ test

Gồm: ví dụ đề, biên (một tài khoản, email trùng trong cùng tài khoản, trùng tên nhưng khác người, chuỗi bắc cầu dài, tất cả cùng một người), phản ví dụ cho các lời giải sai, đối chiếu 4 cách bằng `assert` **sau khi chuẩn hóa** (vì thứ tự nhiều đáp án đúng), test ngẫu nhiên có seed đúng ràng buộc (có các người trùng tên), và test kích thước lớn.

In [24]:
ALL_METHODS = {
    "brute": merge_brute,
    "uf": merge_uf,
    "dfs": merge_dfs,
    "bfs": merge_bfs,
    "leetcode": Solution().accountsMerge,
}

def run_all(accounts, expected=None):
    ref = norm(merge_brute([a[:] for a in accounts]))
    for name, fn in ALL_METHODS.items():
        got = fn([a[:] for a in accounts])
        assert norm(got) == ref, (name, accounts, got, ref)
        assert is_valid_result(accounts, got), (name, accounts, got)
    if expected is not None:
        assert ref == norm(expected), (accounts, ref, expected)
    return ref

fixed_cases = [
    (ex1_in, ex1_out),
    (ex2_in, ex2_out),
    ([["A", "x@m"]], [["A", "x@m"]]),                                          # một tài khoản
    ([["A", "x@m", "x@m"]], [["A", "x@m"]]),                                   # email trùng trong cùng tài khoản
    ([["A", "x@m"], ["A", "x@m"]], [["A", "x@m"]]),                            # hai tài khoản y hệt
    ([["A", "x@m"], ["A", "y@m"]], [["A", "x@m"], ["A", "y@m"]]),              # trùng tên nhưng KHÁC người
    ([["A", "a", "b"], ["A", "b", "c"], ["A", "c", "d"], ["A", "d", "e"]], [["A", "a", "b", "c", "d", "e"]]),   # bắc cầu
    ([["A", "a"], ["A", "b"], ["A", "a", "b"]], [["A", "a", "b"]]),            # tài khoản cuối nối hai tài khoản trước
    ([["A", "c"], ["A", "b", "c"], ["A", "a", "b"]], [["A", "a", "b", "c"]]),  # thứ tự đảo ngược
    ([["A", "b"], ["B", "a"], ["C", "c"]], [["A", "b"], ["B", "a"], ["C", "c"]]),
    ([["David", "David0@m.co", "David1@m.co"], ["David", "David3@m.co", "David4@m.co"], ["David", "David4@m.co", "David5@m.co"],
      ["David", "David2@m.co", "David3@m.co"], ["David", "David1@m.co", "David2@m.co"]],
     [["David", "David0@m.co", "David1@m.co", "David2@m.co", "David3@m.co", "David4@m.co", "David5@m.co"]]),
]
for accounts, expected in fixed_cases:
    run_all(accounts, expected)
print("test cố định:", len(fixed_cases), "bộ x", len(ALL_METHODS), "cách: OK")

test cố định: 11 bộ x 5 cách: OK


In [25]:
# Phản ví dụ 1: gộp theo TÊN (hai người khác nhau có thể trùng tên)
def wrong_by_name(accounts):
    g = defaultdict(set)
    for name, *emails in accounts:
        g[name].update(emails)
    return [[name] + sorted(es) for name, es in g.items()]

acc = [["A", "x@m"], ["A", "y@m"]]
print("gộp theo tên:", wrong_by_name(acc), "| đúng:", norm(merge_uf(acc)))

# Phản ví dụ 2: gộp MỘT lượt, mỗi tài khoản chỉ nối vào nhóm đầu tiên nó chạm
def wrong_single_pass(accounts):
    groups = []                                 # list các [tên, set email]
    for name, *emails in accounts:
        es = set(emails)
        for g in groups:
            if g[1] & es:
                g[1] |= es
                break                           # SAI: chỉ gộp vào nhóm đầu tiên, bỏ các nhóm khác cũng chạm
        else:
            groups.append([name, es])
    return [[n] + sorted(es) for n, es in groups]

acc = [["A", "a", "b"], ["A", "c", "d"], ["A", "b", "c"]]
print("một lượt:", wrong_single_pass(acc), "| đúng:", norm(merge_uf(acc)))

gộp theo tên: [['A', 'x@m', 'y@m']] | đúng: [('A', 'x@m'), ('A', 'y@m')]
một lượt: [['A', 'a', 'b', 'c'], ['A', 'c', 'd']] | đúng: [('A', 'a', 'b', 'c', 'd')]


In [26]:
# Vét cạn tất cả các cấu hình nhỏ: 3 tài khoản, mỗi tài khoản là tập con KHÔNG RỖNG của {e0, e1, e2, e3} (một tên duy nhất)
from itertools import product, combinations

emails_pool = ["e0", "e1", "e2", "e3"]
subsets = [list(c) for r in range(1, 5) for c in combinations(emails_pool, r)]        # 15 tập con
cnt = 0
for trio in product(subsets, repeat=3):
    accounts = [["N"] + list(s) for s in trio]
    run_all(accounts)
    cnt += 1
print("vét cạn", cnt, "cấu hình 3 tài khoản: mọi cách khớp và hợp lệ")

# thêm: 4 tài khoản trên {e0,e1,e2}
subsets3 = [list(c) for r in range(1, 4) for c in combinations(emails_pool[:3], r)]    # 7 tập con
cnt = 0
for quad in product(subsets3, repeat=4):
    run_all([["N"] + list(s) for s in quad])
    cnt += 1
print("vét cạn", cnt, "cấu hình 4 tài khoản: OK")

vét cạn

 3375 cấu hình 3 tài khoản: mọi cách khớp và hợp lệ
vét cạn 2401 cấu hình 4 tài khoản: OK


In [27]:
# Ngẫu nhiên có seed, đúng ràng buộc: NHIỀU người, một số trùng tên; email của người khác nhau không giao nhau;
# mỗi người có vài tài khoản, mỗi tài khoản là một tập con ngẫu nhiên các email của người đó.
rng = random.Random(721)
names = ["John", "Mary", "Gabe", "Kim"]

def random_accounts(rng, n_people, max_emails_per_person, max_accounts_per_person):
    accounts = []
    next_email = 0
    for _ in range(n_people):
        name = rng.choice(names)
        pool = [f"{name.lower()}{next_email + k}@m.co" for k in range(rng.randint(1, max_emails_per_person))]
        next_email += len(pool)
        for _ in range(rng.randint(1, max_accounts_per_person)):
            k = rng.randint(1, min(9, len(pool)))                 # tối đa 9 email => độ dài tài khoản <= 10
            accounts.append([name] + rng.sample(pool, k))
    rng.shuffle(accounts)
    return accounts

for _ in range(1500):
    accounts = random_accounts(rng, rng.randint(1, 4), 6, 4)
    run_all(accounts)
print("ngẫu nhiên nhỏ: OK")

# Đã biết nhóm "thật" ở mức người: mỗi người có một tài khoản BAO TRÙM + các tài khoản con => kết quả phải đúng bằng số người
for _ in range(300):
    n_people = rng.randint(1, 5)
    accounts, truth = [], []
    next_email = 0
    for p in range(n_people):
        name = rng.choice(names)
        pool = [f"p{p}_{k}@m.co" for k in range(rng.randint(1, 9))]
        truth.append([name] + sorted(pool))
        # các tài khoản con sao cho đồ thị liên thông: tài khoản t chứa email t và t+1 (chuỗi), nên các email nối nhau bắc cầu
        if len(pool) == 1:
            accounts.append([name, pool[0]])
        else:
            for k in range(len(pool) - 1):
                accounts.append([name, pool[k], pool[k + 1]])
    rng.shuffle(accounts)
    for name, fn in ALL_METHODS.items():
        assert norm(fn([a[:] for a in accounts])) == norm(truth), name
print("ngẫu nhiên có đáp án biết trước: OK")

ngẫu nhiên nhỏ: OK
ngẫu nhiên có đáp án biết trước: OK


In [28]:
# Kích thước lớn nhất: 1000 tài khoản, mỗi tài khoản <= 9 email
rng = random.Random(7210)
accounts = []
next_id = 0
while len(accounts) < 1000:
    name = rng.choice(names)
    size = rng.randint(1, 40)
    pool = [f"{name.lower()}{next_id + k}@m.co" for k in range(size)]
    next_id += size
    for _ in range(rng.randint(1, 60)):
        if len(accounts) >= 1000:
            break
        accounts.append([name] + rng.sample(pool, rng.randint(1, min(9, size))))
rng.shuffle(accounts)
assert len(accounts) == 1000 and all(2 <= len(a) <= 10 for a in accounts)

ref = norm(merge_uf(accounts))
assert is_valid_result(accounts, merge_uf(accounts))
for name in ["dfs", "bfs", "leetcode"]:
    assert norm(ALL_METHODS[name]([a[:] for a in accounts])) == ref, name
print("1000 tài khoản ->", len(ref), "người: các cách khớp nhau")

# Trường hợp xấu: một chuỗi bắc cầu dài 1000 tài khoản (mỗi tài khoản nối với tài khoản kế)
chain = [["Z", f"z{i}@m.co", f"z{i + 1}@m.co"] for i in range(999)]
chain.append(["Z", "z999@m.co"])
rng.shuffle(chain)
res = Solution().accountsMerge(chain)
assert len(res) == 1 and len(res[0]) == 1 + 1000
assert norm(merge_dfs(chain)) == norm(merge_bfs(chain)) == norm(res)
print("chuỗi bắc cầu dài: một người, 1000 email: OK")

1000 tài khoản -> 34 người: các cách khớp nhau
chuỗi bắc cầu dài: một người, 1000 email: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Gộp theo **tên**

Đề nhấn mạnh: hai tài khoản **trùng tên** vẫn có thể là hai người khác nhau. Chỉ **email chung** mới chứng minh cùng một người. Đã thấy ở mục 12: `wrong_by_name` gộp `["A","x@m"]` và `["A","y@m"]` thành một người.

### Lỗi 2 — Chỉ gộp một lượt (thiếu bắc cầu)

Mỗi tài khoản có thể chạm **nhiều nhóm khác nhau** cùng lúc; khi đó phải gộp **tất cả** các nhóm đó lại, không chỉ nhóm đầu tiên. `wrong_single_pass` ở mục 12 để lại hai nhóm đáng lẽ phải là một. Union-Find/DFS/BFS xử lý bắc cầu tự nhiên.

### Lỗi 3 — Dùng `parent[i]` thay vì `find(i)` để lấy gốc

Sau nhiều lần `union`, `parent[i]` của một nút **chưa chắc là gốc** (đặc biệt khi chưa nén đường đi). Gom nhóm theo `parent[i]` sẽ tách một người thành nhiều nhóm. Luôn dùng `find(i)`.

In [29]:
def wrong_no_find(accounts):
    uf = UnionFind(len(accounts))
    d = {}
    for i, (_, *emails) in enumerate(accounts):
        for e in emails:
            if e in d:
                # nối thô: gốc của i làm con của gốc của d[e] (không union by size) để dễ thấy chuỗi cha
                a, b = uf.find(i), uf.find(d[e])
                if a != b:
                    uf.p[a] = b
            else:
                d[e] = i
    g = defaultdict(set)
    for i, (_, *emails) in enumerate(accounts):
        g[uf.p[i]].update(emails)                 # SAI: dùng p[i] thay vì find(i)
    return [[accounts[r][0]] + sorted(es) for r, es in g.items()]

acc = [["A", "e1"], ["A", "e2"], ["A", "e1", "e2"]]
print("dùng parent[i]:", norm(wrong_no_find(acc)))
print("dùng find(i)  :", norm(merge_uf(acc)))

dùng parent[i]: [('A', 'e1', 'e2'), ('A', 'e1', 'e2')]
dùng find(i)  : [('A', 'e1', 'e2')]


### Lỗi 4 — Quên sắp xếp hoặc loại trùng email

Đề yêu cầu email **theo thứ tự đã sắp xếp**. Nếu dùng `list(set(...))` (thứ tự không xác định) hoặc `list` có trùng (khi hai tài khoản cùng chứa một email) thì sai định dạng. Dùng `sorted(set_of_emails)`.

In [30]:
def wrong_unsorted(accounts):
    uf = UnionFind(len(accounts))
    d = {}
    for i, (_, *emails) in enumerate(accounts):
        for e in emails:
            if e in d:
                uf.union(i, d[e])
            else:
                d[e] = i
    g = defaultdict(list)
    for i, (_, *emails) in enumerate(accounts):
        g[uf.find(i)].extend(emails)             # SAI: dùng list -> trùng email, chưa sắp xếp
    return [[accounts[r][0]] + es for r, es in g.items()]

print("sai :", wrong_unsorted(ex1_in)[0])
print("đúng:", [a for a in merge_uf(ex1_in) if len(a) == 4][0])

sai : ['John', 'johnsmith@mail.com', 'john_newyork@mail.com', 'johnsmith@mail.com', 'john00@mail.com']
đúng: ['John', 'john00@mail.com', 'john_newyork@mail.com', 'johnsmith@mail.com']


### Lỗi 5 — So sánh kết quả mà không chuẩn hóa thứ tự

Đề cho phép trả các nhóm theo **thứ tự bất kỳ**; so sánh trực tiếp `result == expected` có thể báo sai dù kết quả đúng. Hãy chuẩn hóa: `sorted(tuple(a) for a in result)`.

In [31]:
got = Solution().accountsMerge(ex1_in)
print("so sánh trực tiếp:", got == ex1_out, "| sau chuẩn hóa:", norm(got) == norm(ex1_out))
print("thứ tự trả về của lời giải:", [a[0] + ":" + str(len(a) - 1) for a in got])

so sánh trực tiếp: True | sau chuẩn hóa: True
thứ tự trả về của lời giải: ['John:3', 'Mary:1', 'John:1']


## 14. Mẹo & nhận dạng pattern

- **"Nếu A liên quan B và B liên quan C thì A, B, C cùng nhóm"** → bài **thành phần liên thông**: Union-Find, DFS hoặc BFS.
- **Union-Find** hợp khi **các quan hệ đến dần dần** và ta chỉ hỏi "cùng nhóm không?"; viết ngắn, gần `O(1)` mỗi thao tác.
- Mẹo `d[email] = tài khoản đầu tiên`: nối mọi tài khoản chung email qua **một đại diện**, tránh so từng cặp.
- Khi gom kết quả theo nhóm, **luôn dùng `find(i)`** (gốc thật), không dùng `parent[i]`.
- Nếu đề nói "thứ tự bất kỳ" → khi tự kiểm thử, **chuẩn hóa** (sắp xếp) rồi mới so sánh, hoặc viết hàm kiểm tra tính hợp lệ.
- Họ bài liên quan: *Number of Provinces* (547), *Number of Connected Components in an Undirected Graph* (323), *Redundant Connection* (684), *Satisfiability of Equality Equations* (990), *Smallest String With Swaps* (1202).

## 15. Biến thể

### 15.1. Chỉ đếm số người

Số người = số thành phần liên thông = `N - số lần union thật`.

### 15.2. Trả về chỉ số tài khoản của từng người

Thay vì gộp email, trả về nhóm các **chỉ số** `i` (sắp xếp tăng).

### 15.3. Người có nhiều email nhất

Lấy nhóm có số email lớn nhất (hòa: tên nhỏ nhất theo từ điển, rồi email đầu tiên).

### 15.4. Email không phân biệt hoa thường

Chuẩn hóa `email.lower()` trước khi dùng làm khóa (nhưng giữ nguyên bản gốc hoặc nêu rõ quy ước trong kết quả).

In [32]:
def count_people(accounts):
    return count_effective_unions(accounts)[2]

def groups_of_indices(accounts):
    uf = UnionFind(len(accounts))
    d = {}
    for i, (_, *emails) in enumerate(accounts):
        for e in emails:
            if e in d:
                uf.union(i, d[e])
            else:
                d[e] = i
    g = defaultdict(list)
    for i in range(len(accounts)):
        g[uf.find(i)].append(i)
    return sorted(sorted(v) for v in g.values())

def richest_person(accounts):
    people = merge_uf(accounts)
    return max(people, key=lambda p: (len(p) - 1, [-ord(c) for c in p[0]]))

print(count_people(ex1_in), groups_of_indices(ex1_in))
print(richest_person(ex1_in))

# kiểm chứng bằng oracle
rng = random.Random(15)
for _ in range(500):
    accounts = random_accounts(rng, rng.randint(1, 4), 6, 4)
    ref = merge_brute(accounts)
    assert count_people(accounts) == len(ref)
    gi = groups_of_indices(accounts)
    assert sum(len(g) for g in gi) == len(accounts) and len(gi) == len(ref)
    # mỗi nhóm chỉ số phải khớp với một nhóm email của oracle
    for g in gi:
        emails = sorted({e for i in g for e in accounts[i][1:]})
        assert tuple([accounts[g[0]][0]] + emails) in set(norm(ref))
print("các biến thể 15.1-15.3 OK")

3 [[0, 1], [2], [3]]
['John', 'john00@mail.com', 'john_newyork@mail.com', 'johnsmith@mail.com']
các biến thể 15.1-15.3 OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Gộp lặp lại | Gộp hai nhóm có giao tới khi hết | `O(N^3 E)` | `O(K)` | Chỉ để hiểu đề / oracle |
| 2. Union-Find | `email → tài khoản đầu`, union khi trùng | `O(K α + K log K)` | `O(K + N)` | **Nộp bài**, ngắn, quan hệ đến dần dần |
| 3. DFS email | Đồ thị email, thành phần liên thông | `O(K log K)` | `O(K)` | Khi quen đồ thị, không cần cấu trúc Union-Find |
| 4. BFS tài khoản-email | Đồ thị hai phía, lan theo email | `O(K log K)` | `O(K + N)` | Cần duyệt theo tầng/khoảng cách, hoặc tránh đệ quy |

In [33]:
rng = random.Random(16)
accounts = random_accounts(rng, 60, 15, 12)
accounts = accounts[:1000]
answers = {name: norm(fn([a[:] for a in accounts])) for name, fn in ALL_METHODS.items() if name != "brute"}
print({name: len(v) for name, v in answers.items()}, "(số người)")
assert len({tuple(v) for v in answers.values()}) == 1

{'uf': 62, 'dfs': 62, 'bfs': 62, 'leetcode': 62} (số người)


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
uf = UnionFind(5)
uf.union(0, 1); uf.union(3, 4)
print(uf.find(0) == uf.find(1), uf.find(1) == uf.find(3))
name, *rest = ["Bob", "a@x", "b@x"]
print(name, rest, len(rest))
print(sorted({"b@x", "a@x", "b@x"}))
```

In [34]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `accounts = [["A","a","b"],["A","c"],["A","b","c"],["B","d"]]`, hãy cho biết có bao nhiêu **người** và liệt kê kết quả đã chuẩn hóa. Hỏi: tài khoản 0 và tài khoản 1 có chung email không? Vì sao vẫn cùng một người?

In [35]:
# Bài 2: điền kết quả tính tay
hand_people = None          # số người
hand_result = None          # list các tài khoản kết quả, đã sắp xếp (như norm)

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành `find` (có nén đường đi) và `union` của Union-Find.

In [36]:
class ExerciseUF:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, x):
        # TODO 1: nếu p[x] khác x thì gán p[x] = find(p[x]) (nén đường đi)
        # TODO 2: trả về p[x]
        pass

    def union(self, a, b):
        # TODO 3: lấy gốc ra = find(a), rb = find(b)
        # TODO 4: nếu khác nhau thì gán p[ra] = rb và trả True, ngược lại trả False
        pass


# uf = ExerciseUF(5)
# assert uf.union(0, 1) is True and uf.union(1, 0) is False
# uf.union(3, 4)
# assert uf.find(0) == uf.find(1) and uf.find(0) != uf.find(3)

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 3, hãy tự viết `exercise_4_dfs(accounts)` bằng DFS trên đồ thị email (hoặc BFS), trả về kết quả hợp lệ (thứ tự tùy ý).

In [37]:
def exercise_4_dfs(accounts: List[List[str]]) -> List[List[str]]:
    # TODO
    pass


# assert norm(exercise_4_dfs(ex1_in)) == norm(ex1_out)
# assert norm(exercise_4_dfs(ex2_in)) == norm(ex2_out)
# assert norm(exercise_4_dfs([["A", "x"], ["A", "y"]])) == norm([["A", "x"], ["A", "y"]])

### Bài 5 — Mini challenge: nhóm các **chỉ số** tài khoản (khó)

Viết `exercise_5_groups(accounts)` trả về list các nhóm chỉ số: mỗi nhóm là danh sách chỉ số (tăng dần) của các tài khoản thuộc cùng một người; các nhóm sắp xếp theo chỉ số nhỏ nhất của nhóm.

```python
exercise_5_groups([["A","a","b"],["A","c"],["A","b","c"],["B","d"]]) -> [[0, 1, 2], [3]]
exercise_5_groups([["A","x"],["A","y"],["A","x"]])                    -> [[0, 2], [1]]
```

In [38]:
def exercise_5_groups(accounts: List[List[str]]) -> List[List[int]]:
    # TODO
    pass


# assert exercise_5_groups([["A", "a", "b"], ["A", "c"], ["A", "b", "c"], ["B", "d"]]) == [[0, 1, 2], [3]]
# assert exercise_5_groups([["A", "x"], ["A", "y"], ["A", "x"]]) == [[0, 2], [1]]
# assert exercise_5_groups([["A", "x"]]) == [[0]]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `union` nối hai tập; `find` trả gốc.
- Mức 2: `0, 1` cùng tập; `3, 4` cùng tập; hai tập này khác nhau.
- Mức 3: dòng 1 `True False`; dòng 2 `Bob ['a@x', 'b@x'] 2`; dòng 3 `['a@x', 'b@x']` (set loại trùng).

**Bài 2**
- Mức 1: tài khoản 2 chứa `b` (chung với 0) và `c` (chung với 1).
- Mức 2: 0 và 1 không chung email trực tiếp (`{a,b}` và `{c}`), nhưng cùng liên thông qua tài khoản 2.
- Mức 3: 2 người: `("A","a","b","c")` và `("B","d")`.

**Bài 3**
- Mức 1: giống Union-Find ở mục 3.5, bỏ phần `size`.
- Mức 2: `find`: `if self.p[x] != x: self.p[x] = self.find(self.p[x])`.
- Mức 3: `union`: `ra, rb = self.find(a), self.find(b)`; nếu `ra == rb` trả `False`; ngược lại `self.p[ra] = rb`, trả `True`.

**Bài 4**
- Mức 1: nối email đầu của mỗi tài khoản với các email còn lại (hai chiều).
- Mức 2: lưu `owner[email] = tên`; DFS từ mỗi email chưa thăm, gom thành phần.
- Mức 3: kết quả mỗi thành phần: `[owner[start]] + sorted(comp)`.

**Bài 5**
- Mức 1: dùng cách 2 nhưng gom **chỉ số** thay vì email.
- Mức 2: `g[uf.find(i)].append(i)`; duyệt `i` tăng dần nên mỗi list tự tăng dần.
- Mức 3: cuối cùng `sorted(g.values(), key=lambda v: v[0])`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [39]:
uf = UnionFind(5)
uf.union(0, 1); uf.union(3, 4)
print(uf.find(0) == uf.find(1), uf.find(1) == uf.find(3))
name, *rest = ["Bob", "a@x", "b@x"]
print(name, rest, len(rest))
print(sorted({"b@x", "a@x", "b@x"}))

assert (uf.find(0) == uf.find(1), uf.find(1) == uf.find(3)) == (True, False)
assert (name, rest, len(rest)) == ("Bob", ["a@x", "b@x"], 2)
assert sorted({"b@x", "a@x", "b@x"}) == ["a@x", "b@x"]

True False
Bob ['a@x', 'b@x'] 2
['a@x', 'b@x']


### Đáp án Bài 2

In [40]:
acc = [["A", "a", "b"], ["A", "c"], ["A", "b", "c"], ["B", "d"]]
res = norm(Solution().accountsMerge(acc))
print(res)
assert len(res) == 2
assert res == [("A", "a", "b", "c"), ("B", "d")]
assert not ({"a", "b"} & {"c"})        # 0 và 1 không chung email trực tiếp

[('A', 'a', 'b', 'c'), ('B', 'd')]


### Đáp án Bài 3

In [41]:
class ExerciseUFSolution:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        self.p[ra] = rb
        return True

uf = ExerciseUFSolution(5)
assert uf.union(0, 1) is True and uf.union(1, 0) is False
uf.union(3, 4)
assert uf.find(0) == uf.find(1) and uf.find(0) != uf.find(3)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [42]:
def exercise_4_solution(accounts: List[List[str]]) -> List[List[str]]:
    graph = defaultdict(list)
    owner = {}
    for name, *emails in accounts:
        for e in emails:
            owner[e] = name
            graph[emails[0]].append(e)
            graph[e].append(emails[0])
    seen, out = set(), []
    for start in owner:
        if start in seen:
            continue
        seen.add(start)
        stack, comp = [start], []
        while stack:
            u = stack.pop()
            comp.append(u)
            for v in graph[u]:
                if v not in seen:
                    seen.add(v)
                    stack.append(v)
        out.append([owner[start]] + sorted(comp))
    return out

assert norm(exercise_4_solution(ex1_in)) == norm(ex1_out)
assert norm(exercise_4_solution(ex2_in)) == norm(ex2_out)
assert norm(exercise_4_solution([["A", "x"], ["A", "y"]])) == norm([["A", "x"], ["A", "y"]])
rng = random.Random(44)
for _ in range(300):
    acc = random_accounts(rng, rng.randint(1, 4), 6, 4)
    assert norm(exercise_4_solution(acc)) == norm(merge_brute(acc))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [43]:
def exercise_5_solution(accounts: List[List[str]]) -> List[List[int]]:
    uf = UnionFind(len(accounts))
    d = {}
    for i, (_, *emails) in enumerate(accounts):
        for e in emails:
            if e in d:
                uf.union(i, d[e])
            else:
                d[e] = i
    g = defaultdict(list)
    for i in range(len(accounts)):
        g[uf.find(i)].append(i)
    return sorted(g.values(), key=lambda v: v[0])

assert exercise_5_solution([["A", "a", "b"], ["A", "c"], ["A", "b", "c"], ["B", "d"]]) == [[0, 1, 2], [3]]
assert exercise_5_solution([["A", "x"], ["A", "y"], ["A", "x"]]) == [[0, 2], [1]]
assert exercise_5_solution([["A", "x"]]) == [[0]]
rng = random.Random(55)
for _ in range(300):
    acc = random_accounts(rng, rng.randint(1, 4), 6, 4)
    assert exercise_5_solution(acc) == sorted(groups_of_indices(acc), key=lambda v: v[0])
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template Union-Find

```python
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))
        self.size = [1] * n

    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])       # nén đường đi
        return self.p[x]

    def union(self, a, b):
        pa, pb = self.find(a), self.find(b)
        if pa == pb:
            return False
        if self.size[pa] > self.size[pb]:
            self.p[pb] = pa; self.size[pa] += self.size[pb]
        else:
            self.p[pa] = pb; self.size[pb] += self.size[pa]
        return True


class Solution:
    def accountsMerge(self, accounts: List[List[str]]) -> List[List[str]]:
        uf = UnionFind(len(accounts))
        d = {}                                          # email -> tài khoản đầu tiên chứa nó
        for i, (_, *emails) in enumerate(accounts):
            for email in emails:
                if email in d:
                    uf.union(i, d[email])
                else:
                    d[email] = i
        g = defaultdict(set)
        for i, (_, *emails) in enumerate(accounts):
            g[uf.find(i)].update(emails)                # gom theo GỐC THẬT
        return [[accounts[root][0]] + sorted(emails) for root, emails in g.items()]
```

Câu thần chú: **"chung email thì nối; gom theo gốc `find`; email `sorted` kèm tên."**

### Checklist trước khi submit

- [ ] Gộp theo **email chung** (bắc cầu), tuyệt đối không gộp theo tên.
- [ ] Dùng `find(i)` (không phải `parent[i]`) khi gom nhóm.
- [ ] Dùng `set` để loại trùng, `sorted` để sắp xếp email; tên lấy từ một tài khoản bất kỳ trong nhóm.
- [ ] Khi tự kiểm thử, **chuẩn hóa thứ tự** (sắp xếp các nhóm) trước khi so sánh.
- [ ] Test: một tài khoản, email trùng trong cùng tài khoản, trùng tên khác người, chuỗi bắc cầu, thứ tự tài khoản đảo ngược.
- [ ] Skeleton đúng: `class Solution`, method `accountsMerge(self, accounts: List[List[str]]) -> List[List[str]]`; lớp phụ `UnionFind` đặt phía trên `Solution`.